In [3]:
import xml.etree.ElementTree as ET

import re


file_path = r"C:\Users\user\Downloads\assignment v7a.twb"


tree = ET.parse(file_path)

root = tree.getroot()


# Step 1: Build a map of internal name -> caption

# e.g. "[Calculation_1820855438614528]" -> "[Monthly Income]"

id_to_name = {}

for col in root.iter('column'):

    internal_name = col.get('name', '')      # e.g. [Calculation_1820855438614528]

    caption = col.get('caption', '').strip() # e.g. "Monthly Income"

    if internal_name and caption:

        id_to_name[internal_name] = caption


def resolve_formula(formula):

    """Replace all [Calculation_XXXXX] with their human-readable names."""

    def replacer(match):

        key = match.group(0)  # e.g. [Calculation_1820855438614528]

        return f"[{id_to_name.get(key, key)}]"

    return re.sub(r'\[Calculation_\w+\]', replacer, formula)


# Step 2: Extract and print with resolved names

seen = set()

results = []


bad_caption_patterns = re.compile(

    r'(THEN|ELSE|END|AVG|SUM|IF |ELSEIF|\[Calculation_|>=|<=|POWER|FIXED|MIN\(|MAX\()',

    re.IGNORECASE

)


for col in root.iter('column'):

    caption = col.get('caption', '').strip()

    calc = col.find('calculation')

    if not caption or calc is None:

        continue

    if bad_caption_patterns.search(caption):

        continue

    if caption.startswith('[') or caption.startswith('*'):

        continue

    formula = calc.get('formula', '').strip()

    if not formula or caption in seen:

        continue

    seen.add(caption)

    resolved = resolve_formula(formula)

    results.append((caption, resolved))


# Print

print(f"{'Field Name':<45} | Formula")

print("-" * 120)

for name, formula in sorted(results):

    lines = formula.split('\n')

    print(f"{name:<45} | {lines[0].strip()}")

    for line in lines[1:]:

        if line.strip():

            print(f"{'':<45}   {line.strip()}")

    print()


print(f"\nTotal unique calculated fields: {len(results)}")

Field Name                                    | Formula
------------------------------------------------------------------------------------------------------------------------
Affordability Category                        | IF [Mortgage Burden Ratio] >= 0.40 THEN "Crisis (>40%)"
                                                ELSEIF [Mortgage Burden Ratio] >= 0.30 THEN "Cost-Burdened (30-40%)"
                                                ELSEIF [Mortgage Burden Ratio] >= 0.20 THEN "Stretched (20-30%)"
                                                ELSE "Affordable (<20%)"
                                                END

Affordability Ratio                           | [median_household_income_lagged] / [home_value]

Bar - National Average                        | AVG([home_value])

Bar - Selected State                          | AVG(IF [state] = [Parameters].[Parameter 2] THEN [home_value] ELSE NULL END)

Burden Filter Combined                        | [Top 20 Burden States] OR